# 03 — Log `claims_fe` Pyfunc & Deploy to Model Serving

Logs the FE pyfunc via `mlflow.pyfunc.log_model()`, bundling the wheel from
notebook 02 as a code artifact. Registers to Unity Catalog, then deploys a
provisioned serving endpoint.

**Why bundle the wheel via `code_paths`** instead of referencing it from a Volume
at build time: Model Serving builds a container at deploy time, and the `/Volumes/`
FUSE mount is not reliably available inside the container build environment. Using
`code_paths` ships the `.whl` directly inside the MLflow model artifact, making the
deployment fully self-contained with no external path dependencies.

**Why `scale_to_zero_enabled=False`**: pandas + numpy cold-start is 10–30s per
replica. Tight-SLA use cases can't afford that. Use provisioned concurrency from
the start.

In [0]:
%pip install mlflow /Volumes/fins_genai/classic_ml/claims_fe_wheels/claims_fe-0.1.0-py3-none-any.whl --force-reinstall
dbutils.library.restartPython()

Processing /Volumes/fins_genai/classic_ml/claims_fe_wheels/claims_fe-0.1.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [0]:
CATALOG = "fins_genai"
SCHEMA = "classic_ml"
WHEEL_VERSION = "0.1.0"
WHEEL_VOLUME = "claims_fe_wheels"
WHEEL_FILENAME = f"claims_fe-{WHEEL_VERSION}-py3-none-any.whl"
WHEEL_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{WHEEL_VOLUME}/{WHEEL_FILENAME}"

MODEL_NAME = "claims_fe_transformer"
REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
ENDPOINT_NAME = "claims-fe-transformer"
TABLE_NAME = "fe_test_payloads"

SHIM_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{WHEEL_VOLUME}"
SHIM_PATH = f"{SHIM_DIR}/fe_shim.py"

## Write the pyfunc shim to a Volume

This is the `python_model` file handed to `log_model`. It imports from the installed
wheel and registers an instance via `mlflow.models.set_model()`. Keeps the MLflow
artifact tiny — the actual code lives in the wheel.

In [0]:
SHIM_CONTENT = '''\
from claims_fe.transformer import ClaimFeatureTransformer
import mlflow

mlflow.models.set_model(ClaimFeatureTransformer())
'''

with open(SHIM_PATH, "w") as f:
    f.write(SHIM_CONTENT)
print(f"Shim written: {SHIM_PATH}")

Shim written: /Volumes/fins_genai/classic_ml/claims_fe_wheels/fe_shim.py


## Log the model

In [0]:
import mlflow
import pandas as pd
from mlflow.models import ModelSignature
from mlflow.types.schema import Schema, ColSpec

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').first()[0]}/claims-fe-transformer")

input_schema = Schema([ColSpec("string", "payload_json")])
output_schema = Schema([ColSpec("string", "features_json")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Use one real synthetic payload as the input example
sample_row = (
    spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")
    .select("payload_json")
    .limit(1)
    .collect()[0]
)
input_example = pd.DataFrame({"payload_json": [sample_row.payload_json]})

pip_requirements = [
    "mlflow>=3.0",
    "pandas>=1.5",
    "numpy>=1.23",
    # Wheel is bundled via code_paths -> resolves as code/<filename> in the artifact
    f"code/{WHEEL_FILENAME}",
]

with mlflow.start_run(run_name="claims_fe_transformer_v0_1_0") as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=SHIM_PATH,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
        code_paths=[WHEEL_PATH],
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    print(f"Model URI: {model_info.model_uri}")
    print(f"Run ID:    {run.info.run_id}")

print(f"Registered: {REGISTERED_MODEL_NAME}")

2026/04/26 21:02:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/278973608200451/models/m-9a2f1488376c4d23ac2c4fc92502ea4a?o=1444828305810485
/local_disk0/.ephemeral_nfs/envs/pythonEnv-e0d8494c-4989-4166-a561-91b7b4779a30/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/04/26 21:02:01 INFO mlflow.pyfunc: Validating input example against model signature
Registered model 'fins_genai.classic_ml.claims_fe_transformer' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

🔗 Created version '5' of model 'fins_genai.classic_ml.claims_fe_transformer': https://e2-demo-field-eng.cloud.databricks.com/explore/data/models/fins_genai/classic_ml/claims_fe_transformer/version/5?o=1444828305810485


Model URI: models:/m-9a2f1488376c4d23ac2c4fc92502ea4a
Run ID:    2c73fb1d240e462cab40aa163f80ca90
Registered: fins_genai.classic_ml.claims_fe_transformer


## Deploy to Model Serving

In [0]:
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput, TrafficConfig, Route
from databricks.sdk.errors import ResourceDoesNotExist
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()

versions = list(w.model_versions.list(full_name=REGISTERED_MODEL_NAME))
latest_version = str(max(int(v.version) for v in versions))
print(f"Deploying {REGISTERED_MODEL_NAME} v{latest_version} -> endpoint '{ENDPOINT_NAME}'")

served_entities = [
    ServedEntityInput(
        entity_name=REGISTERED_MODEL_NAME,
        entity_version=latest_version,
        workload_size="Small",
        scale_to_zero_enabled=False,
    )
]

traffic_config = TrafficConfig(
    routes=[
        Route(
            served_model_name=f"{MODEL_NAME}-{latest_version}",
            traffic_percentage=100,
        )
    ]
)

served_config = EndpointCoreConfigInput(
    name=ENDPOINT_NAME,
    served_entities=served_entities,
    traffic_config=traffic_config,
)

def wait_for_config_update(endpoint_name, timeout=120, interval=5):
    for _ in range(int(timeout / interval)):
        ep = w.serving_endpoints.get(endpoint_name)
        if not ep.state.config_update:
            return True
        time.sleep(interval)
    return False

try:
    w.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_entities=served_entities,
        traffic_config=traffic_config,
    )
    print(f"Updated endpoint '{ENDPOINT_NAME}' to v{latest_version}")
    if not wait_for_config_update(ENDPOINT_NAME):
        print("Warning: config_update did not clear after update. Route optimization may not be active yet.")
except ResourceDoesNotExist:
    w.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=served_config,
        route_optimized=True,
    )
    print(f"Created endpoint '{ENDPOINT_NAME}'")
    if not wait_for_config_update(ENDPOINT_NAME):
        print("Warning: config_update did not clear after creation. Route optimization may not be active yet.")

Deploying fins_genai.classic_ml.claims_fe_transformer v5 -> endpoint 'claims-fe-transformer'
Created endpoint 'claims-fe-transformer'


## Endpoint status
Re-run until `State: READY`. Container build + provisioned-concurrency warm-up
typically takes ~10–15 min for a pyfunc with a wheel install.

In [0]:
ep = w.serving_endpoints.get(ENDPOINT_NAME)
print(f"State:         {ep.state.ready}")
print(f"Config update: {ep.state.config_update}")
if ep.pending_config and ep.pending_config.served_entities:
    for e in ep.pending_config.served_entities:
        print(f"  Pending: {e.name} -> {e.state}")

State:         EndpointStateReady.READY
Config update: EndpointStateConfigUpdate.NOT_UPDATING
